# Out-of-scheme labels 

The classification was repeated keeping the labels exactly as the model returned them, so that the
invalid ones could be counted before any remapping. These are the numbers reported in §3.3 ("for DA,
30 labels (0.61%) were remapped ... for CMA, 73 labels (1.48%)").

* **Input:** `validation/unmapped_labels/<coding scheme>.csv`
* **Output:** `results/tables/out_of_scheme_labels.csv`
* **Check:** out-of-scheme labels stay around 1% of the utterances and are lexical variants of valid labels

Only the Dialogue act and the Conversational Management schemes were re-run this way; the Epistemic
Stance scheme had 11 invalid labels, all lexical variants, checked directly on the labels themselves.

In [1]:
import pandas as pd
from pathlib import Path
import os
import json

# Dataset to work on: 'data' is the study data, 'data_example' the synthetic example.
DATA = Path('..') / os.environ.get('DATASET_FOLDER', 'data')

CODING_SCHEMES = ['lin_dialogue_acts',
                  'del_bosque_cma']

## Count the labels the classifier invented

Pure spelling variants of a valid label are normalised first, and are not counted: the example prompt
of the Conversational Management scheme spells "Tutor Floor Initiating or Floor **Maintaining**" while
`tags.json` spells it "**Mantaining**", so the model returns that spelling in 912 utterances. The same
normalisation is applied in `01_few_shot_classification`.

In [2]:
SPELLING_VARIANTS = {
    'del_bosque_cma': {
        'Tutor Floor Initiating or Floor Maintaining': 'Tutor Floor Initiating or Floor Mantaining',
        'Tutor Floor Initiating or Floor Mantining': 'Tutor Floor Initiating or Floor Mantaining',
    },
    'acosta_epistemic_stance': {
        'Other acts that maintaint discussion': 'Other acts that maintain discussion',
    },
}

rows = []
for coding_scheme in CODING_SCHEMES:

    df = pd.read_csv(DATA / 'validation' / 'unmapped_labels' / (coding_scheme + '.csv'))

    # Read coding scheme
    path = Path('..') / 'coding_schemes' / coding_scheme
    with open(path / 'tags.json') as f:
        scheme = json.load(f)

    # 1. Build a flat set of all allowed tags (unique)
    allowed_tags = set()
    for tags_list in scheme.values():
        allowed_tags.update(tags_list)

    # 2. Get counts of each label in the DataFrame
    label = df['label'].astype(str).str.strip().replace(SPELLING_VARIANTS.get(coding_scheme, {}))
    label_counts = label.value_counts()

    # 3. Identify labels that are not allowed
    invalid_labels = label_counts[~label_counts.index.isin(allowed_tags)]

    # 4. Report results
    print('===', coding_scheme, ':', len(df), 'utterances ===')
    if invalid_labels.empty:
        print('  No conflicts')
    for label_name, count in invalid_labels.items():
        print('  {}: {} rows ({:.2f}%)'.format(label_name, count, 100 * count / len(df)))
        rows.append({'coding_scheme': coding_scheme, 'label': label_name, 'n': int(count),
                     'pct_of_utterances': round(100 * count / len(df), 2)})
    print('  total: {} ({:.2f}%)\n'.format(invalid_labels.sum(), 100 * invalid_labels.sum() / len(df)))

out_of_scheme = pd.DataFrame(rows)

path_out = DATA / 'results' / 'tables'
path_out.mkdir(parents=True, exist_ok=True)
out_of_scheme.to_csv(path_out / 'out_of_scheme_labels.csv', index=False)

out_of_scheme

=== lin_dialogue_acts : 4936 utterances ===
  Acknowledgement: 24 rows (0.49%)
  Yes-No Question: 3 rows (0.06%)
  Confirmation: 3 rows (0.06%)
  Example Question: 1 rows (0.02%)
  total: 31 (0.63%)

=== del_bosque_cma : 4936 utterances ===
  Tutor Acknowledge Backchannel: 64 rows (1.30%)
  Tutor Acknowledge Response: 4 rows (0.08%)
  Tutor Acknowledge: 3 rows (0.06%)
  Tutor Floor Maintaining: 2 rows (0.04%)
  total: 73 (1.48%)



,coding_scheme,label,n,pct_of_utterances
0,lin_dialogue_acts,Acknowledgement,24,0.49
1,lin_dialogue_acts,Yes-No Question,3,0.06
2,lin_dialogue_acts,Confirmation,3,0.06
3,lin_dialogue_acts,Example Question,1,0.02
4,del_bosque_cma,Tutor Acknowledge Backchannel,64,1.30
5,del_bosque_cma,Tutor Acknowledge Response,4,0.08
6,del_bosque_cma,Tutor Acknowledge,3,0.06
7,del_bosque_cma,Tutor Floor Maintaining,2,0.04


§3.3 reports 30 remapped labels and 3 assigned to "Other" for the Dialogue act Scheme, and 73 remapped
and 13 assigned to "Other" for the Conversational Management scheme. This check is a re-run of the
classifier, so its counts differ by one or two labels from the ones recorded while the corpus was
being labelled.

## The labels kept in the analysed files

After the remapping of `01_few_shot_classification`, every label belongs to the coding scheme or to
"Other".

In [3]:
for coding_scheme in ['lin_dialogue_acts', 'acosta_epistemic_stance', 'del_bosque_cma']:

    path = Path('..') / 'coding_schemes' / coding_scheme
    with open(path / 'tags.json') as f:
        scheme = json.load(f)

    tags = []
    for e in scheme:
        tags += scheme[e]
    tags += ['Other']

    df = pd.read_csv(DATA / 'labeled_conversations' / (coding_scheme + '.csv'))

    labels = df['label'].unique()
    print('Checking labels:', coding_scheme)
    problemas = False
    for label in labels:
        if label not in tags:
            print(' ', label)
            problemas = True

    if problemas:
        print('  The classification process invented labels')
    else:
        print('  No conflicts -', (df['label'] == 'Other').sum(), 'utterances labelled Other')

Checking labels: lin_dialogue_acts
  No conflicts - 3 utterances labelled Other
Checking labels: acosta_epistemic_stance
  No conflicts - 0 utterances labelled Other
Checking labels: del_bosque_cma
  No conflicts - 4 utterances labelled Other
